# Topic modelling with BERTopic (sentence level)

Trains the final BERTopic model on the sentences of the Lesefassung and exports the results.

**Input**
- `input/lesefassung.txt` – plain text produced by `01_preprocessing` (`3_tei_to_text.xsl`); paragraphs are delimited by `///`, sentences by `###`
- `stopwords/stopwords_edited.txt` – edited French stopword list, used for the topic representations only

**Output**
- `output/processed_sentences_lesefassung.txt` – the 10,039 sentences as fed into the model, with paragraph numbers
- `models/final_2026-09-27.mm` – the fitted model (saved *before* the stopword update, see below)
- `output/topic_info_with_weights_output_schwerin_final_2026-09-27.json` – topics with top terms and c-TF-IDF weights
- `output/document_info_output_schwerin_final_2026-09-27.json` – topic assignment per sentence
- `output/bertopic_results_with_paragraphs_schwerin_final_2026-09-27.json` – as above, plus paragraph number
- `barchart_visualization_schwerin_final_2026-09-27.html` – bar chart of the top terms per topic

**Environment**: Google Colab, Python 3.13, T4 GPU, package versions in `requirements.txt`.
With `language="french"` BERTopic uses the embedding model `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.

**Reproducibility**: all random seeds are fixed (`RANDOM_STATE = 42`). Results on a different GPU or with different package versions may differ slightly.

Run the notebook with this folder as working directory; all paths are relative.

## Setup

In [ ]:
# Install the pinned versions (uncomment if needed)
# %pip install -r requirements.txt

# In Google Colab: mount Drive and change into the folder containing this notebook
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/<path-to>/02_bertopic

In [ ]:
from bertopic import BERTopic
import pandas as pd

# Prepare data

Splits the text into sentences at `###` and records the paragraph number of each sentence (paragraphs are split at `///`).

In [ ]:
import re

file_path = './input/lesefassung.txt'
output_file_path = './output/processed_sentences_lesefassung.txt'

with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

def clean_text(sentence):
    # Replace non-breaking spaces and other non-standard spaces
    sentence = sentence.replace('\xa0', ' ').replace('\n', ' ')
    # Replace multiple spaces with a single space
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    return sentence

def process_text_for_bertopic(text):
    # Split the text into paragraphs using '///' as the delimiter
    paragraphs = text.split('///')
    sentences = []
    sentence_paragraph_map = []

    # Enumerate the paragraphs and split sentences at '###'
    for idx, paragraph in enumerate(paragraphs):
        # Trim whitespace which might be left from splitting
        trimmed_paragraph = paragraph.strip()
        if trimmed_paragraph:  # Ensure the paragraph is not empty
            # Split the paragraph into sentences using '###'
            paragraph_sentences = [s.strip() for s in trimmed_paragraph.split('###') if s.strip()]
            # Clean and extend the sentences to the main list
            cleaned_sentences = [clean_text(sentence) for sentence in paragraph_sentences]
            sentences.extend(cleaned_sentences)
            # Store the paragraph number for each sentence for later analysis
            sentence_paragraph_map.extend([idx + 1] * len(cleaned_sentences))

    return sentences, sentence_paragraph_map

# Process the text for BERTopic
sentences, sentence_paragraph_map = process_text_for_bertopic(text)

# Write sentences and paragraph numbers to an output file
with open(output_file_path, 'w', encoding='utf-8') as output_file:
    output_file.write(f"The text contains {len(sentences)} sentences based on the '###' split.\n\n")
    for sentence, paragraph_num in zip(sentences, sentence_paragraph_map):
        output_file.write(f"Paragraph {paragraph_num}: {sentence}\n")

# Print the first 100 sentences to the console
print("The first 100 sentences:")
for i, (sentence, paragraph_num) in enumerate(zip(sentences, sentence_paragraph_map)):
    if i >= 100:  # Stop after printing 100 sentences
        break
    print(f"Paragraph {paragraph_num}: {sentence}")

print(f"All sentences have been written to {output_file_path}.")

# Create topics

In [ ]:
import random
import numpy as np
import torch
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# -----------------------------------------------------
# 1. Fix all random seeds
# -----------------------------------------------------
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# -----------------------------------------------------
# 2. Define deterministic subcomponents
# -----------------------------------------------------
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# -----------------------------------------------------
# 3. Define BERTopic (no random_state parameter here)
# -----------------------------------------------------
topic_model = BERTopic(
    verbose=True,
    language="french",
    umap_model=umap_model,
    hdbscan_model=hdbscan_model
)

# -----------------------------------------------------
# 4. Fit and transform
# -----------------------------------------------------
topics, probs = topic_model.fit_transform(sentences)

# Save the model

The model is saved right after fitting, i.e. before the stopword update below. To reuse the published model instead of training, run *Prepare data*, skip *Create topics* and *Save the model*, and continue from the load cell.

In [ ]:
topic_model.save("./models/final_2026-09-27.mm")

In [ ]:
topic_model = BERTopic.load("./models/final_2026-09-27.mm")

# Inspect

Shows the embedding model used.

In [ ]:
print(topic_model.embedding_model.embedding_model)
print(topic_model.embedding_model.embedding_model._modules['0'].auto_model.config._name_or_path)

# Finetuning the model

Recomputes the topic representations with the edited stopword list. This only changes the terms describing each topic, not the assignment of sentences to topics.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Load the edited French stopword list

with open('./stopwords/stopwords_edited.txt', 'r', encoding='utf-8') as file:
  stop_words = [word.strip() for word in file.readlines()]

vectorizer_model = CountVectorizer(stop_words=stop_words)
topic_model.update_topics(sentences, vectorizer_model=vectorizer_model)

# Investigate the topics

In [ ]:
topic_model.get_topic_info()

In [ ]:
import json

# Retrieve the topic info DataFrame
topic_info_df = topic_model.get_topic_info()

# Retrieve weights for each topic
topics_with_weights = {}
for topic_id in topic_info_df["Topic"]:
    if topic_id != -1:  # Skip the catch-all topic (-1) if not needed
        # Retrieve terms and their weights for the topic
        topics_with_weights[topic_id] = topic_model.get_topic(topic_id)

# Combine the topic info with the weights
topic_info_with_weights = []
for _, row in topic_info_df.iterrows():
    topic_id = row["Topic"]
    row_data = row.to_dict()
    if topic_id in topics_with_weights:
        # Add terms and weights for the topic
        row_data["Terms_with_Weights"] = topics_with_weights[topic_id]
    topic_info_with_weights.append(row_data)

# Convert the combined data to a JSON string
topic_info_with_weights_json = json.dumps(
    topic_info_with_weights, ensure_ascii=False, indent=4
)

# Specify the output file path
output_file = './output/topic_info_with_weights_output_schwerin_final_2026-09-27.json'

# Save the JSON data to a file
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(topic_info_with_weights_json)

print(f"Topic info with weights has been saved to {output_file}")

# Topic labels and document info

In [ ]:
topic_labels = topic_model.generate_topic_labels(nr_words=20, topic_prefix=False, separator=" - ")
topic_model.set_topic_labels(topic_labels)
# topic_model.set_topic_labels({2: "Lineages"})

In [ ]:
# Retrieve the document info DataFrame based on sentences
document_info_df = topic_model.get_document_info(sentences)

# Convert the DataFrame to a JSON string
document_info_json = document_info_df.to_json(orient='records', force_ascii=False, indent=4)

# Specify the output file path
output_file = './output/document_info_output_schwerin_final_2026-09-27.json'

# Save the JSON data to a file
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(document_info_json)

print(f"Document info has been saved to {output_file}")

# Visualize

In [ ]:
# Save the visualization as an HTML file
fig = topic_model.visualize_barchart(width=280, height=330, top_n_topics=114, n_words=10)
fig.write_html("barchart_visualization_schwerin_final_2026-09-27.html")

# Map documents to paragraphs and generate mapped Output JSON

In [ ]:
import json

# File paths
bertopic_output_file = './output/document_info_output_schwerin_final_2026-09-27.json'
output_file = './output/bertopic_results_with_paragraphs_schwerin_final_2026-09-27.json'

# Load BERTopic output from the JSON file
with open(bertopic_output_file, 'r', encoding='utf-8') as f:
    bertopic_results = json.load(f)


# Ensure that we are mapping the correct sentences from BERTopic to their paragraph numbers
# Assuming that the documents in `bertopic_results` match the sentences exactly

# We will now enhance the BERTopic results by adding paragraph numbers
for idx, result in enumerate(bertopic_results):
    if idx < len(sentence_paragraph_map):
        # Add the corresponding paragraph number from the map
        paragraph_num = sentence_paragraph_map[idx]
        result['Paragraph'] = paragraph_num
    else:
        result['Paragraph'] = None  # Just in case there are more results than sentences

# Save the enhanced results with paragraph mapping back to a new JSON file
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(bertopic_results, f, ensure_ascii=False, indent=4)

print(f"BERTopic results with paragraph mapping saved to {output_file}")

# Optional: topic hierarchy

Exploratory only, not saved.

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(sentences)

In [ ]:
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)